# EmpowerLens - sequential fine-tuning: PatternReframe -> Annotated

**A separate experiment from `kaggle_runner_patternreframe.ipynb`.** That notebook
MERGES the two datasets into one training set. This one trains on them one after the
other. Same data, different way of combining it.

### Why sequential should beat merging

Merging puts 8,712 PatternReframe rows and 1,278 Annotated rows in the same batches,
so the model averages two label conventions and two text-length distributions at
once, with the borrowed data outnumbering the target 7:1.

Sequential splits the job in two:

| stage | trains on | learns |
|---|---|---|
| A | PatternReframe alone (7,846) | what the ten distortion *patterns* look like |
| B | Annotated alone (1,278), from A's weights | this dataset's labelling convention and real text lengths |

The classifier head is **kept, not reinitialised**. Both stages are 10-way multilabel
over the same `ml_*` columns in the same order, so stage B refines the head A already
trained. That is the whole advantage over merging - and it only works because the two
label spaces line up exactly.

Partial forgetting of PatternReframe is *desirable*. Its 17-word one-liners and its
label convention are not the target; its notion of what each pattern looks like is.
A lower LR in stage B keeps the representations while re-aligning the head.

This is intermediate-task transfer - Phang et al. 2018 ("STILTs"), Pruksachatkun et
al. 2020 - the standard move when the target set is small and a larger, noisier,
related set exists.

### Baselines to beat (Annotated test, 3 seeds)

| | macro_f1 |
|---|---|
| **Stage 2 isolated, Annotated only** | **0.277 +/- 0.016** |
| flat multilabel, Annotated only | 0.237 +/- 0.030 |
| cascade end-to-end | 0.240 +/- 0.012 |

0.277 is the one that counts - it is distorted-only, exactly like this experiment.

### What could go wrong

1. **Length shift.** PatternReframe is a median 17 words, Annotated 129. Stage A
   learns from one-liners. This is the likeliest failure mode.
2. **`emotional_reasoning` has zero PatternReframe coverage** - 9 of 10 classes get
   pretrained, one does not. Its per-class F1 is the control.
3. **Catastrophic forgetting** if stage B's LR is too high. Hence 1e-5 with LLRD.

### The in-domain diagnostic

Stage A is scored **twice**: on the Annotated test set (out of domain, the number
that matters) and on 866 held-out PatternReframe rows (in domain). Without the
second, a low stage-A score is ambiguous - it could mean stage A learned nothing, or
that it learned fine and did not transfer, and those need opposite responses.

| in-domain | on Annotated | reading |
|---|---|---|
| high | low | domain gap - stage A worked, transfer is the problem |
| low | low | stage A itself failed - fix it before concluding anything |

The held-out rows are the authors' own **valid** split. That matters: the official
splits are persona-disjoint (231/115/812 personas, zero pairwise overlap) and each
thought is written *from* its persona, so a random slice would put the same persona
on both sides and read optimistically. Reserving their *test* split would be more
conventional but costs 6,120 rows and would leave less training data than Annotated
already provides.

**The evaluation target never moves:** the untouched Annotated test set.

In [ ]:
import os
os.chdir('/kaggle/working/')
!rm -rf /kaggle/working/empowerlens

REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "nayab-space"

!git clone --branch $BRANCH $REPO_URL empowerlens
os.chdir('/kaggle/working/empowerlens')

!pip install --upgrade pip setuptools wheel
!pip install -q -r requirements-transformer.txt
!pip install -q sentencepiece protobuf

In [ ]:
# mental/mental-roberta-base is GATED. Accept its licence at
# https://huggingface.co/mental/mental-roberta-base while logged in, then attach the
# HF_TOKEN secret to THIS notebook: Add-ons -> Secrets -> tick HF_TOKEN.
# Secrets are per-notebook even though the token itself is account-level.
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face Hub.")
except Exception as e:
    print(f"[warn] no working HF_TOKEN secret ({e}) - training will 401 until this is set.")

In [ ]:
# Pin one GPU BEFORE any subprocess imports torch, then load the bootstrap.
# sequential_bootstrap.py execs cascade_bootstrap.py itself, so this brings in
# sh() (with its pipe-drain fix), sync(), the timeouts, AND run_stage().
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!nvidia-smi -L

SEQBOOT = '/kaggle/working/empowerlens/notebooks/sequential_bootstrap.py'
exec(open(SEQBOOT).read())


## 1 - Build stage A's training set

Stage A trains on PatternReframe **alone**, but it still needs a val set for epoch
selection and early stopping. Its own official split is out of domain for that
purpose, so `--eval-from` takes val/test from `data/splits_stage2` and leaves train
as PatternReframe only. **Both stages therefore select epochs against the same val
set** - the real distribution - which is what you want.

The official split is not discarded entirely, though. It is **persona-disjoint**
(231/115/812 personas, zero pairwise overlap), and `--holdout official-valid`
reserves the authors' valid split - 866 rows after mapping - as an in-domain
diagnostic. Those rows leave the training pool, which is why stage A trains on 7,846
rather than 8,712.

This builds **three** dirs, and the cell below prints all three. The 2.4 MB tarball
is committed to the repo, so nothing is downloaded.

In [ ]:
SEQBOOT = '/kaggle/working/empowerlens/notebooks/sequential_bootstrap.py'
exec(open(SEQBOOT).read())

# --holdout official-valid reserves the authors' own valid split (961 rows, 115
# personas) as an IN-DOMAIN diagnostic. The official splits share ZERO personas, and
# each thought is written FROM its persona, so a random slice would put the same
# persona on both sides and read optimistically. Costs 961 training rows; their TEST
# split would cost 6,807 and leave less data than Annotated already has.
!python -m src.make_splits_patternreframe --eval-from $TARGET_SPLITS \
    --out $PR_SPLITS --holdout official-valid --force

import pandas as pd
print()
for name, p in [("A: " + PR_SPLITS, PR_SPLITS), ("B: " + TARGET_SPLITS, TARGET_SPLITS),
                ("A-diag: " + PR_HOLDOUT, PR_HOLDOUT)]:
    tr = pd.read_csv(f"{p}/train.csv", encoding="utf-8-sig")
    va = pd.read_csv(f"{p}/val.csv",   encoding="utf-8-sig")
    te = pd.read_csv(f"{p}/test.csv",  encoding="utf-8-sig")
    w  = int(tr[tr.columns[0]].astype(str).str.split().str.len().median())
    print(f"  {name:28} train={len(tr):5}  val={len(va)}  test={len(te)}  median_words={w}")

print("\nRows A and B must have IDENTICAL val/test - the eval target never moves.")
print("Row A-diag is DIFFERENT on purpose: its val/test are held-out PatternReframe,")
print("used only for the in-domain diagnostic, never for reporting.")
print("median_words 17 vs ~129 is the distribution shift this experiment is betting against.")

## 2 - Stage A: PatternReframe only

**Scored twice, and both numbers are needed.**

*Out of domain* - the Annotated test set, despite never seeing Annotated training
data. That is **pure zero-shot transfer**: how far borrowed data alone gets you.
Expect it to be poor; its value is as the floor stage B must beat.

*In domain* - 866 held-out, persona-disjoint PatternReframe rows. This is a second
scoring pass over the same checkpoints, no retraining, so it is nearly free.

Read them together. A low out-of-domain score on its own cannot distinguish "stage A
learned nothing" from "stage A learned fine and did not transfer", and those need
opposite responses - fix the run, versus report a domain gap.

2 epochs, not 4: this is an intermediate task, and the goal is usable
representations rather than a converged model on PatternReframe.

In [ ]:
SEQBOOT = '/kaggle/working/empowerlens/notebooks/sequential_bootstrap.py'
exec(open(SEQBOOT).read())

print("=== Stage A: PatternReframe only (7,846 rows; 866 reserved as diagnostic) ===")
for seed in SEEDS:
    run_stage(
        init_model=MODEL, tag=TAG_A, splits_dir=PR_SPLITS,
        out_dir=SEQ_A_OUT, seed=seed, ckpt_root=CK_A,
        extra_flags=("--max-length 256 --truncation head_tail --batch-size 32 "
                     "--epochs 2 --lr 3e-5 --lr-scheduler cosine "
                     "--loss focal --focal-gamma 2.0"),
    )

# Second scoring pass: the SAME checkpoints against held-out PatternReframe. No
# training, so it is cheap. Read it TOGETHER with the numbers above:
#   high here + low above -> domain gap; stage A worked, transfer is the problem
#   low here + low above  -> stage A itself failed; fix it before concluding anything
print("\n=== Stage A, in-domain diagnostic (held-out PatternReframe) ===")
for seed in SEEDS:
    eval_only(stage_a_ckpt(seed), SEQ_A_HOLDOUT_OUT, PR_HOLDOUT, TAG_A, seed)

## 3 - Stage B: continue on Annotated

Starts from stage A's weights, classifier head included, and trains on the 1,278
Annotated distorted rows.

**LR is 1e-5, a third of stage A's.** Too high and stage B simply overwrites what A
learned - the classic catastrophic-forgetting failure, which would show up as stage B
scoring the same as the plain Annotated-only baseline. LLRD on top keeps the lower
encoder layers stabler than the head, which is right when the head is the part that
needs re-aligning to a new label convention.

In [ ]:
SEQBOOT = '/kaggle/working/empowerlens/notebooks/sequential_bootstrap.py'
exec(open(SEQBOOT).read())

print("=== Stage B: continue on Annotated distorted-only (1,278 rows) ===")
for seed in SEEDS:
    ck_a = stage_a_ckpt(seed)
    if not os.path.isdir(ck_a):
        print(f"  [seed {seed}] stage A checkpoint missing at {ck_a} — run cell 2 first")
        continue
    run_stage(
        init_model=ck_a, tag=TAG_B, splits_dir=TARGET_SPLITS,
        out_dir=SEQ_B_OUT, seed=seed, ckpt_root=CK_B,
        extra_flags=("--max-length 256 --truncation head_tail --batch-size 32 "
                     "--epochs 4 --lr 1e-5 --lr-scheduler cosine "
                     "--llrd --llrd-decay 0.9 --early-stopping-patience 2 "
                     "--loss focal --focal-gamma 2.0"),
    )

## 4 - Compare

The ordering that would make this worth writing up:

    stage A (zero-shot)  <  Annotated-only baseline (0.277)  <  stage B

Stage B above 0.277 means the borrowed data genuinely helped. Stage B *below* it means
stage A's representations actively hurt - most likely the 17-vs-129-word shift - and
that is a reportable negative result, not a failed run. Stage B landing exactly on
0.277 means stage A was simply forgotten, so try a lower LR before concluding
anything.

**Before reading any of that, check stage A's in-domain score.** If it is also low,
stage A simply failed and nothing below is interpretable - fix the run first. Only a
*high* in-domain score with a *low* Annotated score licenses the "domain gap"
conclusion.

`emotional_reasoning` is the control: it is the one class PatternReframe could not
pretrain. If the other nine improve and it does not, the gain came from the borrowed
data rather than from extra training steps.

In [ ]:
SEQBOOT = '/kaggle/working/empowerlens/notebooks/sequential_bootstrap.py'
exec(open(SEQBOOT).read())

import pandas as pd
from pathlib import Path

SOURCES = {
    "Annotated only (Stage2 isolated)": "results_stage2",
    "A: PatternReframe only (0-shot)":  SEQ_A_OUT,
    "B: PR -> Annotated (sequential)":  SEQ_B_OUT,
}
frames = []
for label, folder in SOURCES.items():
    p = Path(folder) / "paper_comparison.csv"
    if p.exists():
        d = pd.read_csv(p); d["run"] = label; frames.append(d)
    else:
        print(f"[skip] {p} not found")

if frames:
    allr = pd.concat(frames, ignore_index=True)
    v = allr[(allr.task == "multilabel") & (allr.split == "test")]
    print(v.groupby("run")[["macro_f1", "weighted_f1", "micro_f1"]]
           .agg(["mean", "std"]).round(3).to_string())

print("\n=== emotional_reasoning — the class PatternReframe could NOT pretrain ===")
for label, folder in SOURCES.items():
    vals = []
    for f in Path(folder).glob("per_class_*multilabel*.csv"):
        d = pd.read_csv(f)
        col = "class" if "class" in d.columns else d.columns[0]
        r = d[d[col].astype(str).str.contains("emotional", case=False, na=False)]
        if len(r) and "f1" in d.columns:
            vals.append(float(r.iloc[0]["f1"]))
    if vals:
        print(f"  {label:34} f1 = {sum(vals)/len(vals):.3f}  (mean of {len(vals)} files)")
    else:
        print(f"  {label:34} no per-class CSVs yet")

## 5 - Zip for download

`/kaggle/working` is wiped when the session ends. Do this before closing the tab.

In [ ]:
SEQBOOT = '/kaggle/working/empowerlens/notebooks/sequential_bootstrap.py'
exec(open(SEQBOOT).read())

ZIP = "/kaggle/working/results_sequential.zip"
folders = [f for f in (SEQ_A_OUT, SEQ_B_OUT, SEQ_A_HOLDOUT_OUT)
           if os.path.isdir(f) and os.listdir(f)]
for f in folders:
    sync(f)          # results live inside empowerlens/; zip runs on /kaggle/working
if not folders:
    print("NOTHING TO ZIP.")
else:
    if os.path.exists(ZIP):
        os.remove(ZIP)
    sh("cd /kaggle/working && zip -rq results_sequential.zip " + " ".join(folders))
    print("\nWrote %s (%.1f MB)" % (ZIP, os.path.getsize(ZIP)/1048576))
    print("Download from the Output panel, then locally:")
    print('   Expand-Archive -Path "$env:USERPROFILE\\Downloads\\results_sequential.zip" -DestinationPath . -Force')
    print("   venv\\Scripts\\python.exe -m src.compile_results")